In [1]:
from kafka import KafkaConsumer, errors
import json
import csv
import logging
import os
from datetime import datetime

# Configuración de logs
DIRECTORIO_LOGS = '../transporte/logs_kafka/'
os.makedirs(DIRECTORIO_LOGS, exist_ok=True)
archivo_log = os.path.join(DIRECTORIO_LOGS, 'kafka_consumidor.log')

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(archivo_log, encoding='utf-8')]
)

def log_and_print(mensaje, nivel="info"):
    print(mensaje)
    if nivel == "info":
        logging.info(mensaje)
    elif nivel == "warning":
        logging.warning(mensaje)
    elif nivel == "error":
        logging.error(mensaje)

def consumir_topic_a_csv(nombre_topic, directorio_salida='../transporte/datasets_consumidos/'):
    try:
        consumidor = KafkaConsumer(
            bootstrap_servers='192.168.41.51:9092',
            auto_offset_reset='earliest',
            enable_auto_commit=True,
            value_deserializer=lambda v: json.loads(v.decode('utf-8')),
            consumer_timeout_ms=5000
        )

        log_and_print(f"Iniciando consumo del topico '{nombre_topic}'")

        topics_disponibles = consumidor.topics()
        if nombre_topic not in topics_disponibles:
            log_and_print(f"Topic '{nombre_topic}' no existe", nivel="warning")
            return

        consumidor.subscribe([nombre_topic])
        inicio = datetime.now()
        registros = []

        # Recibir mensajes
        for mensaje in consumidor:
            registros.append(mensaje.value)
            if len(registros) % 10000 == 0:
                log_and_print(f"{len(registros)} registros consumidos de {nombre_topic}")

        fin = datetime.now()
        duracion = (fin - inicio).total_seconds()

        if len(registros) == 0:
            log_and_print(f"El topico '{nombre_topic}' esta vacio (0 registros en {duracion:.2f} segundos)", nivel="warning")
            return

        # Crear CSV solo si hay registros
        os.makedirs(directorio_salida, exist_ok=True)
        ruta_salida = os.path.join(directorio_salida, f"{nombre_topic}.csv")

        with open(ruta_salida, mode='w', newline='', encoding='utf-8') as archivo_csv:
            escritor = csv.DictWriter(archivo_csv, fieldnames=registros[0].keys(), extrasaction='ignore')
            escritor.writeheader()
            for registro in registros:
                escritor.writerow(registro)

        log_and_print(f"Consumo completado del topic '{nombre_topic}' ({len(registros)} registros en {duracion:.2f} segundos)")

    except errors.NoBrokersAvailable:
        log_and_print("Error: No se puede conectar con el broker Kafka.", nivel="error")
    except Exception as e:
        log_and_print(f"Error al consumir el topic '{nombre_topic}': {e}", nivel="error")
    finally:
        try:
            consumidor.close()
        except:
            pass

if __name__ == "__main__":
    for topic in ['InfoMarcas', 'Personas', 'ModelosCoches', 'CochesVentas']:
        consumir_topic_a_csv(topic)
    log_and_print("Finalizó el proceso de consumo de todos los topics.", nivel="info")

Iniciando consumo del topico 'InfoMarcas'
Consumo completado del topic 'InfoMarcas' (40 registros en 5.18 segundos)
Iniciando consumo del topico 'Personas'
10000 registros consumidos de Personas
20000 registros consumidos de Personas
Consumo completado del topic 'Personas' (23906 registros en 25.47 segundos)
Iniciando consumo del topico 'ModelosCoches'
10000 registros consumidos de ModelosCoches
20000 registros consumidos de ModelosCoches
30000 registros consumidos de ModelosCoches
40000 registros consumidos de ModelosCoches
Consumo completado del topic 'ModelosCoches' (47523 registros en 208.05 segundos)
Iniciando consumo del topico 'CochesVentas'
10000 registros consumidos de CochesVentas
20000 registros consumidos de CochesVentas
30000 registros consumidos de CochesVentas
40000 registros consumidos de CochesVentas
50000 registros consumidos de CochesVentas
60000 registros consumidos de CochesVentas
70000 registros consumidos de CochesVentas
80000 registros consumidos de CochesVentas